In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [37]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [78]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()
len(docs)

9

In [79]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_doc = splitter.split_documents(docs)
len(splitted_doc)

26

In [5]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

c:\Users\Armaan Dogra\OneDrive\Desktop\GenAI_Series\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1745.69it/s]


In [80]:
vector_store = InMemoryVectorStore.from_documents(
    documents=splitted_doc,
    embedding=embeddings
)

In [81]:
same_record = vector_store.similarity_search("Patient name")
same_record

[Document(id='feea1b92-1581-4376-8f9c-87bd7da24d7f', metadata={'producer': 'PDFsharp 6.1.0', 'creator': 'Dr Lal PathLabs Limited', 'creationdate': '2025-07-11T09:10:48+00:00', 'author': 'Dr Lal PathLabs Limited', 'title': 'Patient Report', 'subject': 'Lal Pathlabs Report', 'keywords': 'Dr Lal PathLabs Limited', 'moddate': '2025-07-11T09:10:48+00:00', 'source': '../data/medical_report.pdf', 'total_pages': 9, 'page': 6, 'page_label': '7'}, page_content='MD, Pathology\nChief of Laboratory                            \nDr Lal PathLabs Ltd\nDr Kiran Bhargava Pathak\nMD, Pathology\nChief of Laboratory                            \nDr Lal PathLabs Ltd\n*474764803*\n.\nPage 7 of 9'),
 Document(id='4bd00308-1352-4fd1-873e-c68f51bff91e', metadata={'producer': 'PDFsharp 6.1.0', 'creator': 'Dr Lal PathLabs Limited', 'creationdate': '2025-07-11T09:10:48+00:00', 'author': 'Dr Lal PathLabs Limited', 'title': 'Patient Report', 'subject': 'Lal Pathlabs Report', 'keywords': 'Dr Lal PathLabs Limited', 'mod

In [83]:
@tool
def retriever_tool(query:str):
    """
    This tool can help you to retrive the relevant data of the pdf documents and these document 
    have details about medical records.
    """
    print("tool called",query)
    docs = vector_store.similarity_search(query=query,k=4)
    context=""

    for doc in docs:
        context = doc.page_content + "\n\n"
    return context

In [84]:
retriever_tool.invoke("patient name")

tool called patient name


'Consultant- Hemato-Oncopathology & \nFlowcytometry                                     \nNRL - Dr Lal PathLabs Ltd\nDr Sunanda\nMD, Pathology\nSr. Consultant Pathologist - \nHematology & Immunology                              \nNRL - Dr Lal PathLabs Ltd\n-------------------------------End of report --------------------------------\nAHEEEHAPMKHJBKHNLKIHBLLCBILLJCECCJLCIKPLPKEDFBFAPPAHEEEHA\nBNFFFNBPAPBOACIGFGELNGAPAOAHFHAKAKNOBCJKBLEKMCJGNPBNFFFNB\nCIEGCAFJLPNBKNNOPFIAAJHFJGDEHEDFLKPHENFLMLKIOEFLBKHDEHANP\nDJECMEFNBMHIEDINMIMELFMFAFPLBLAFJIFFOAPAKCLJPDNLIJFBKEMEK\nKDCILHFJONFFDIAJMBICKPJDCENJKPCHKKMBACOIPLKGKPNGKFFFOIOND\nFGCBBEFNNICGDKHKILIGCIPBADHFOFAINCFCBKDKBLIKOMNALNEJOGEKD\nDPIMIJFNELODDJHIEKMPCGFHIHAFJBALPPMPBLMIKJCHKJNKIJNFMDILD\nNJJJAMFMAKNKPELJGANHDOMILLBMFMBHIFCCAMNNIIKNOPNKBNFMBHILL\nNKIHJAFMDKOGFAAMFNOIHMIKHEJFEKFBJFFKBKOFOOBIOCNKDJPHNEKLJ\nNMDBPJFCEHNJJDCLCDBFGPNFIDFKPNJHKEPLELPLNKCKIGEOCIGKHMEJP\nICIKOLFLGKCBMMCMPNBDAJEEINMEFCBBJMFEBFOEONDMKPFKEPCIGKAMG\n\n'

In [85]:
llm = ChatGroq(model='openai/gpt-oss-20b')

In [86]:
promt= """
    You are a helpful assistant and provide answers based on the retireved context. 
    always use the "retiever_tool" tool for questions requiring external knowledge.
"""

In [87]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=promt
)

In [88]:
query = "what is patient name and doctors name"
response = agent.invoke({"messages":[{"role":"user","content":query}]})

tool called patient name doctor name
tool called patient name


In [89]:
response["messages"][-1].content

'From the excerpt you provided:\n\n- **Doctor’s name:** Dr\u202fSunanda (MD, Pathology, Sr. Consultant Pathologist – Hematology & Immunology, NRL – Dr\u202fLal\u202fPathLabs Ltd)  \n- **Patient’s name:** The patient’s name is not present in the portion of the report you shared. If you have another section of the document that includes the patient’s details, feel free to paste it and I can locate the name for you.'